### Primero tomamos un pdf y extraemos el texto

In [40]:
import pdfplumber
def extraer_texto_pdf(ruta_pdf):
    texto = "" 
    with pdfplumber.open(ruta_pdf) as pdf:
        for pagina in pdf.pages:
            contenido = pagina.extract_text()
            if contenido: texto += contenido + "\n"
    return texto # Ejemplo de uso
    
ruta = "transformers_teoria.pdf" 
texto = extraer_texto_pdf(ruta)
print(texto)

Redes Neuronales Transformers: Explicación
Teórica
Los Transformers son una arquitectura de redes neuronales introducida en 2017 en el artículo
“Attention is All You Need”.
Se utilizan principalmente en tareas de procesamiento de lenguaje natural, aunque también se
aplican en visión y otras áreas.
A diferencia de modelos anteriores como las redes recurrentes (RNN) o LSTM, los Transformers
no procesan la información de manera secuencial.
En su lugar, utilizan un mecanismo llamado “atención” que les permite considerar todas las partes
de la entrada al mismo tiempo.
El concepto clave de los Transformers es la atención, específicamente la “self-attention” o
autoatención.
Este mecanismo permite que cada palabra en una secuencia evalúe la importancia de las demás
palabras dentro del mismo contexto.
Gracias a esto, el modelo puede capturar relaciones a largo plazo sin depender de pasos
secuenciales.
La arquitectura básica de un Transformer se compone de dos partes principales: el encoder y el

### Separa la letra de la cancion de fondo

In [42]:
import subprocess
import boto3
import os

archivo = "example3.mp3"
bucket = "music-project-ia"

s3 = boto3.client("s3")

try:
    nombre = os.path.splitext(os.path.basename(archivo))[0]

    # 1. Ejecutar Demucs
    subprocess.run([
        "demucs",
        "--two-stems=vocals",
        archivo
    ], check=True)

    print("Separación completada")

    output_folder = os.path.join("separated", "htdemucs", nombre)

    # 2. Convertir WAV → MP3 antes de subir
    for root, dirs, files in os.walk(output_folder):
        for file in files:

            if file.endswith(".wav"):

                wav_path = os.path.join(root, file)
                mp3_path = wav_path.replace(".wav", ".mp3")

                # convertir con ffmpeg
                subprocess.run([
                    "ffmpeg",
                    "-y",
                    "-i", wav_path,
                    "-b:a", "192k",
                    mp3_path
                ], check=True)

                # subir MP3
                s3_key = f"results/{nombre}/{file.replace('.wav','.mp3')}"

                s3.upload_file(mp3_path, bucket, s3_key)

                print(f"Subido: {mp3_path}")

except Exception as e:
    print("Error:", e)

Separación completada
Subido: separated\htdemucs\example3\no_vocals.mp3
Subido: separated\htdemucs\example3\vocals.mp3


### Extraer la letra de la cancion

In [44]:
import boto3
import whisper
import tempfile

s3 = boto3.client("s3")

obj = s3.get_object(
    Bucket="music-project-ia",
    Key="results/example3/vocals.mp3"
)

audio_bytes = obj["Body"].read()

# crear archivo temporal
with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
    local_file = f.name
    f.write(audio_bytes)

# cargar modelo
model = whisper.load_model("base")

# TRANSCRIBIR CON SEGMENTOS
result = model.transcribe(
    local_file,
    language="en"
)

# texto completo
letra_original = result["text"]
num_lines = len(result["segments"])
print("LETRA COMPLETA:\n")
print(letra_original)

print("\nSEGMENTOS:\n")

# frases + tiempo
for seg in result["segments"]:

    inicio = seg["start"]
    fin = seg["end"]
    texto = seg["text"]
 
    print(f"Inicio: {inicio:.2f}s")
    print(f"Fin: {fin:.2f}s")
    print(f"Texto: {texto}")
    print("-" * 40)

C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


LETRA COMPLETA:

 I don't hate my job I'm gonna sew your hair I'm gonna get it I'm gonna take it all I'm gonna keep it for you Hey, you know how to do it One best name Hachim, Sub-E-E-L-S-N-E M-M-J Hachim, Hachim, Hachim, Hachim, Hachim, Hachim, Hachim, Hachim, Hachim, Hachim M-M-M-J M-M-M-J Jack, Jack, Jack, Jack, Jack, I'm gonna do it I'm gonna get it when I'm fine, sorry Mia, Nettin' outside Mia, Nettin' inside Prettin' the ones who wanna easily fightgrunts I'm gonna flirt with you I'm gonna flirt with you Ha, I'm gonna flirt with you I'm gonna flirt with you I'm gonna flirt with you I'm gonna flirt with you I'm gonna anyway I'm gonna meet a new man I'm gonna flirt with you He's gonna give it a shit I'm gonnamod your umm I'm gonna be funny with your singer I'm so firey, firey, firey, firey, firey, firey Keep on running, I'm running, I feel like ketchup How you there, how you there, how you there? Another trophy, my hands come to you Not too many that can't even come to you My car, m

### Crear la letra con LLM

In [46]:
import boto3
from botocore.exceptions import ClientError

def llamar_qwen(prompt):

    client = boto3.client("bedrock-runtime", region_name="us-east-1")

    # 2. Definir el ID del modelo Qwen
    # Ejemplo para Qwen 2.5 72B Instruct (verifica la versión exacta en tu catálogo)
    model_id = "google.gemma-3-12b-it"

    # 3. Preparar el mensaje
    messages = [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ]

    try:
        # 4. Realizar la llamada usando la Converse API
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "maxTokens": 1000,
                "temperature": 0.7,
                "topP": 0.9
            }
        )

        # 5. Extraer y retornar el texto
        return response['output']['message']['content'][0]['text']

    except ClientError as e:
        return f"Error de cliente: {e}"
    except Exception as e:
        return f"Error inesperado: {e}"
        
# Asegúrate de que estas variables tengan contenido real
prompt = f"""
SISTEMA: Eres un extractor de datos técnicos. 
Tu objetivo es tomar la INFORMACIÓN del PDF y darle el ritmo de la ESTRUCTURA sugerida.

[DATOS TÉCNICOS DEL PDF]
{texto}

[ESTRUCTURA RÍTMICA A SEGUIR]
{letra_original}

[REGLAS CRÍTICAS]
- PROHIBIDO: No uses ninguna palabra de la 'ESTRUCTURA RÍTMICA'. 
- OBLIGATORIO: Solo usa conceptos encontrados en el 'DATOS TÉCNICOS DEL PDF'.
- Si mencionas "Stressed out", "Mama", "Sleep" o "Time", la tarea fallará.
- Genera exactamente {num_lines} líneas.
- No saludes, no expliques, solo entrega la letra educativa.

RESULTADO (CANCIÓN TÉCNICA):
"""

# --- Ejemplo de uso ---
if __name__ == "__main__":
    pregunta =  prompt 
    resultado = llamar_qwen(pregunta)
    print("Respuesta:\n", resultado)

Respuesta:
 Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah

Yeah



In [1]:
resultado = """Yeah, 누가 내 수저 더럽대
Yeah, nuga nae sujeo deoreopdae

I don't care 마이크 잡음 금수저 여럿 패
I don't care maikeu jabeum geumsujeo yeoreot pae

버럭해 잘 못 익은 것들 스테끼 여러 개
beoreokae jal mot igeun geotdeul seutekki yeoreo gae

거듭해서 씹어줄게 스타의 저녁에
geodeupaeseo ssibeojulge seutaui jeonyeoge

World business 핵심
World business haeksim

섭외 1순위 매진
seoboe 1sunwi maejin

많지 않지 이 class 가칠 만끽
manji anji i class gachil mankkik

좋은 향기에 악췬 반칙
joeun hyanggie akchwin banchik

Mic, mic bungee
Mic, mic bungee


Mic, mic bungee
Mic, mic bungee

Bright light 전진
Bright light jeonjin

망할 거 같았겠지만 I'm fine, sorry
manghal geo gatatgetjiman I'm fine, sorry

미안해 Billboard
mianhae Billboard

미안해 worldwide
mianhae worldwide

아들이 넘 잘나가서 미안해 엄마
adeuri neom jallagaseo mianhae eomma

대신해줘 니가 못한 효도
daesinhaejwo niga motan hyodo

우리 콘서트 절대 없어 포도
uri konseoteu jeoldae eopseo podo

I do it, I do it 넌 맛없는 라따뚜이
I do it, I do it neon maseomneun rattattui

혹 배가 아프다면 고소해
hok baega apeudamyeon gosohae

Sue it
Sue it


Did you see my bag? (Where?)
Did you see my bag? (Where?)

Did you see my bag? (Where?)
Did you see my bag? (Where?)

It's hella trophies
It's hella trophies

And it's hella thick (hella thick, hella thick)
And it's hella thick (hella thick, hella thick)

What you think 'bout that? (Well)
What you think 'bout that? (Well)

What you think 'bout that? (Well)
What you think 'bout that? (Well)

I bet it got my haters hella sick (hella sick)
I bet it got my haters hella sick (hella sick)


Come and follow me, follow me, with your signs up
Come and follow me, follow me, with your signs up

I'm so firin', firin', boy, your time's up
I'm so firin', firin', boy, your time's up

Keep on and runnin' and runnin' until I catch up
Keep on and runnin' and runnin' until I catch up

How you dare?
How you dare?

How you dare?
How you dare?

How you dare?
How you dare?


Another trophy
Another trophy

My hands carry 'em
My hands carry 'em

Too many that I can't even count 'em
Too many that I can't even count 'em

Mic drop, mic drop
Mic drop, mic drop

발발 조심 너네 말말 조심
balbal josim neone malmal josim


Somebody stop me
Somebody stop me

I'm 'bouta pop off
I'm 'bouta pop off

Too busy, you know
Too busy, you know

My body ain't enuff
My body ain't enuff

Mic drop, mic drop
Mic drop, mic drop

발발 조심 너네 말말 조심
balbal josim neone malmal josim


Baby, watch your mouth (mouth)
Baby, watch your mouth (mouth)

It come back around ('round)
It come back around ('round)

Once upon a time (time)
Once upon a time (time)

We learnt how to fly (fly)
We learnt how to fly (fly)

Go look at your mirror
Go look at your mirror

Same damn clothes (yeah)
Same damn clothes (yeah)

You know how I feel
You know how I feel

개행복 (turn up)
gaehaengbok (turn up)

How many hours do we fly?
How many hours do we fly?

I keep on dreamin' on the cloud
I keep on dreamin' on the cloud

Yeah, I'm on the mountain
Yeah, I'm on the mountain

Yeah, I'm on the bay
Yeah, I'm on the bay

Everyday we vibin'
Everyday we vibin'

Mic drop, bam!
Mic drop, bam!


Did you see my bag? (Where?)
Did you see my bag? (Where?)

Did you see my bag? (Where?)
Did you see my bag? (Where?)

It's hella trophies
It's hella trophies

And it's hella thick (hella thick, hella thick)
And it's hella thick (hella thick, hella thick)

What you think 'bout that? (Well)
What you think 'bout that? (Well)

What you think 'bout that? (Well)
What you think 'bout that? (Well)

I bet it got my haters hella sick (hella sick)
I bet it got my haters hella sick (hella sick)


Come and follow me, follow me with your signs up
Come and follow me, follow me with your signs up

I'm so firin', firin', boy, your time's up
I'm so firin', firin', boy, your time's up

Keep on and runnin' and runnin' until I catch up
Keep on and runnin' and runnin' until I catch up

How you dare?
How you dare?

How you dare?
How you dare?

How you dare?
How you dare?


Another trophy
Another trophy

My hands carry 'em
My hands carry 'em

Too many that I can't even count 'em
Too many that I can't even count 'em

Mic drop, mic drop
Mic drop, mic drop

발발 조심 너네 말말 조심
balbal josim neone malmal josim


Somebody stop me
Somebody stop me

I'm 'bouta pop off
I'm 'bouta pop off

Too busy, you know
Too busy, you know

My body ain't enuff
My body ain't enuff

Mic drop, mic drop
Mic drop, mic drop

발발 조심 너네 말말 조심
balbal josim neone malmal josim


Haters gon' hate
Haters gon' hate

Players gon' play
Players gon' play

Live a life, man
Live a life, man

Good luck
Good luck


더 볼 일 없어 마지막 인사야
deo bol il eopseo majimak insaya

할 말도 없어 사과도 하지 마
hal maldo eopseo sagwado haji ma

더 볼 일 없어 마지막 인사야
deo bol il eopseo majimak insaya

할 말도 없어 사과도 하지 마
hal maldo eopseo sagwado haji ma


잘 봐 넌 그 꼴 나지
jal bwa neon geu kkol naji

우린 탁 쏴 마치 콜라지
urin tak sswa machi kollaji

너의 각막 깜짝 놀라지
neoui gangmak kkamjjak nollaji

꽤 꽤 폼나지, yeah
kkwae kkwae pomnaji, yeah"""


### Clonar voz

In [2]:
import boto3
from TTS.api import TTS

BUCKET = "music-project-ia"
KEY = "results/example2/vocals.mp3"
LOCAL_AUDIO = "voz.mp3"

generated_lines = [
    x.strip()
    for x in resultado.split("\n")
    if x.strip()
]
s3 = boto3.client("s3")
s3.download_file(BUCKET, KEY, LOCAL_AUDIO)

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

for i, line in enumerate(generated_lines):

    print(f"Generando línea {i}: {line}")

    tts.tts_to_file(
        text=line,
        speaker_wav=LOCAL_AUDIO,
        language="es",
        file_path=f"line_{i}.wav"
    )

C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
Generando línea 0: Yeah, 누가 내 수저 더럽대
 > Text splitted to sentences.
['Yeah, 누가 내 수저 더럽대']
 > Processing time: 25.578052282333374
 > Real-time factor: 3.2684054985248663
Generando línea 1: Yeah, nuga nae sujeo deoreopdae
 > Text splitted to sentences.
['Yeah, nuga nae sujeo deoreopdae']
 > Processing time: 12.525413513183594
 > Real-time factor: 3.1818590779458327
Generando línea 2: I don't care 마이크 잡음 금수저 여럿 패
 > Text splitted to sentences.
["I don't care 마이크 잡음 금수저 여럿 패"]
 > Processing time: 19.161110877990723
 > Real-time factor: 3.137270515472373
Generando línea 3: I don't care maikeu jabeum geumsujeo yeoreot pae
 > Text splitted to sentences.
["I don't care maikeu jabeum geumsujeo yeoreot pae"]
 > Processing time: 15.576578617095947
 > Real-time factor: 3.0838201992077794
Generando línea 4: 버럭해 잘 못 익은 것들 스테끼 여러 개
 > Text splitted to sentences.
['버럭해 잘 못 익은 것들 스테끼 여러 개']
 > Processing time: 

In [3]:
import boto3
from pydub import AudioSegment
import tempfile

# =========================
# S3 CONFIG
# =========================
s3 = boto3.client("s3")

BUCKET = "music-project-ia"
INSTRUMENTAL_KEY = "results/example3/no_vocals.mp3"

# =========================
# DESCARGAR INSTRUMENTAL
# =========================
with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
    instrumental_path = tmp.name

s3.download_file(BUCKET, INSTRUMENTAL_KEY, instrumental_path)
print("Instrumental descargada desde S3")

# =========================
# CARGAR AUDIO
# =========================
instrumental = AudioSegment.from_mp3(instrumental_path)
instrumental = instrumental - 8  # bajar volumen del beat

final = instrumental

# =========================
# MEZCLA CON VOCES
# =========================
max_len = min(len(result["segments"]), len(generated_lines))

for i in range(max_len):

    try:
        voice = AudioSegment.from_wav(f"line_{i}.wav")

        voice = voice + 5  # subir voz

        start_ms = int(result["segments"][i]["start"] * 1000)

        print(f"Overlay línea {i} en {start_ms} ms")

        final = final.overlay(
            voice,
            position=start_ms
        )

    except Exception as e:
        print(f"Error línea {i}: {e}")

# =========================
# EXPORT FINAL
# =========================
final.export("cancion_final.mp3", format="mp3")

print("✅ listo")

Instrumental descargada desde S3


NameError: name 'result' is not defined

In [ ]:
##Todo

In [ ]:
"""import pdfplumber
def extraer_texto_pdf(ruta_pdf):
    texto = "" 
    with pdfplumber.open(ruta_pdf) as pdf:
        for pagina in pdf.pages:
            contenido = pagina.extract_text()
            if contenido: texto += contenido + "\n"
    return texto # Ejemplo de uso
    
ruta = "topologia_explicacion.pdf" 
texto = extraer_texto_pdf(ruta)
print(texto)

import subprocess
import boto3
import os

archivo = "example3.mp3"
bucket = "music-project-ia"

s3 = boto3.client("s3")

try:
    nombre = os.path.splitext(os.path.basename(archivo))[0]

    # 1. Ejecutar Demucs
    subprocess.run([
        "demucs",
        "--two-stems=vocals",
        archivo
    ], check=True)

    print("Separación completada")

    output_folder = os.path.join("separated", "htdemucs", nombre)

    # 2. Convertir WAV → MP3 antes de subir
    for root, dirs, files in os.walk(output_folder):
        for file in files:

            if file.endswith(".wav"):

                wav_path = os.path.join(root, file)
                mp3_path = wav_path.replace(".wav", ".mp3")

                # convertir con ffmpeg
                subprocess.run([
                    "ffmpeg",
                    "-y",
                    "-i", wav_path,
                    "-b:a", "192k",
                    mp3_path
                ], check=True)

                # subir MP3
                s3_key = f"results/{nombre}/{file.replace('.wav','.mp3')}"

                s3.upload_file(mp3_path, bucket, s3_key)

                print(f"Subido: {mp3_path}")

except Exception as e:
    print("Error:", e)

import boto3
import whisper
import tempfile

s3 = boto3.client("s3")

obj = s3.get_object(
    Bucket="music-project-ia",
    Key="results/example3/vocals.mp3"
)

audio_bytes = obj["Body"].read()

# crear archivo temporal
with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
    local_file = f.name
    f.write(audio_bytes)

# cargar modelo
model = whisper.load_model("base")

# TRANSCRIBIR CON SEGMENTOS
result = model.transcribe(
    local_file,
    language="en"
)

# texto completo
letra_original = result["text"]
num_lines = len(result["segments"])
print("LETRA COMPLETA:\n")
print(letra_original)

print("\nSEGMENTOS:\n")

# frases + tiempo
for seg in result["segments"]:

    inicio = seg["start"]
    fin = seg["end"]
    texto = seg["text"]
 
    print(f"Inicio: {inicio:.2f}s")
    print(f"Fin: {fin:.2f}s")
    print(f"Texto: {texto}")
    print("-" * 40)
import boto3
from botocore.exceptions import ClientError

def llamar_qwen(prompt):

    client = boto3.client("bedrock-runtime", region_name="us-east-1")

    # 2. Definir el ID del modelo Qwen
    # Ejemplo para Qwen 2.5 72B Instruct (verifica la versión exacta en tu catálogo)
    model_id = "google.gemma-3-12b-it"

    # 3. Preparar el mensaje
    messages = [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ]

    try:
        # 4. Realizar la llamada usando la Converse API
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "maxTokens": 1000,
                "temperature": 0.7,
                "topP": 0.9
            }
        )

        # 5. Extraer y retornar el texto
        return response['output']['message']['content'][0]['text']

    except ClientError as e:
        return f"Error de cliente: {e}"
    except Exception as e:
        return f"Error inesperado: {e}"
        
# Asegúrate de que estas variables tengan contenido real
prompt = f"""
SISTEMA: Eres un extractor de datos técnicos. 
Tu objetivo es tomar la INFORMACIÓN del PDF y darle el ritmo de la ESTRUCTURA sugerida.

[DATOS TÉCNICOS DEL PDF]
{texto}

[ESTRUCTURA RÍTMICA A SEGUIR]
{letra_original}

[REGLAS CRÍTICAS]
- PROHIBIDO: No uses ninguna palabra de la 'ESTRUCTURA RÍTMICA'. 
- OBLIGATORIO: Solo usa conceptos encontrados en el 'DATOS TÉCNICOS DEL PDF'.
- Si mencionas "Stressed out", "Mama", "Sleep" o "Time", la tarea fallará.
- Genera exactamente {num_lines} líneas.
- No saludes, no expliques, solo entrega la letra educativa.

RESULTADO (CANCIÓN TÉCNICA):
"""

# --- Ejemplo de uso ---
if __name__ == "__main__":
    pregunta =  prompt 
    resultado = llamar_qwen(pregunta)
    print("Respuesta:\n", resultado)

import boto3
from TTS.api import TTS

BUCKET = "music-project-ia"
KEY = "results/example2/vocals.mp3"
LOCAL_AUDIO = "voz.mp3"

generated_lines = [
    x.strip()
    for x in resultado.split("\n")
    if x.strip()
]
s3 = boto3.client("s3")
s3.download_file(BUCKET, KEY, LOCAL_AUDIO)

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

for i, line in enumerate(generated_lines):

    print(f"Generando línea {i}: {line}")

    tts.tts_to_file(
        text=line,
        speaker_wav=LOCAL_AUDIO,
        language="es",
        file_path=f"line_{i}.wav"
    )
import boto3
from pydub import AudioSegment
import tempfile

# =========================
# S3 CONFIG
# =========================
s3 = boto3.client("s3")

BUCKET = "music-project-ia"
INSTRUMENTAL_KEY = "results/example3/no_vocals.mp3"

# =========================
# DESCARGAR INSTRUMENTAL
# =========================
with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
    instrumental_path = tmp.name

s3.download_file(BUCKET, INSTRUMENTAL_KEY, instrumental_path)
print("Instrumental descargada desde S3")

# =========================
# CARGAR AUDIO
# =========================
instrumental = AudioSegment.from_mp3(instrumental_path)
instrumental = instrumental - 8  # bajar volumen del beat

final = instrumental

# =========================
# MEZCLA CON VOCES
# =========================
max_len = min(len(result["segments"]), len(generated_lines))

for i in range(max_len):

    try:
        voice = AudioSegment.from_wav(f"line_{i}.wav")

        voice = voice + 5  # subir voz

        start_ms = int(result["segments"][i]["start"] * 1000)

        print(f"Overlay línea {i} en {start_ms} ms")

        final = final.overlay(
            voice,
            position=start_ms
        )

    except Exception as e:
        print(f"Error línea {i}: {e}")

# =========================
# EXPORT FINAL
# =========================
final.export("cancion_final.mp3", format="mp3")

print("✅ listo") """"